In [1]:
import pandas as pd
from pathlib import Path
from typing import Any
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

data_dir = Path("/home/komputer7/SparkProjects/online-retail-pyspark/data")
csv_files = sorted(data_dir.rglob("*.csv"))
if not csv_files:
	raise FileNotFoundError(f"No CSV files found under {data_dir}")

dataset_path = csv_files[0]
print(f"Using dataset: {dataset_path}")
df = pd.read_csv(dataset_path)

Using dataset: /home/komputer7/SparkProjects/online-retail-pyspark/data/staging/online_retail.csv


In [2]:
# Keep only valid product descriptions and non-return quantities.
df = df.dropna(subset=["Description"])
df = df[df["Quantity"] > 0]

# Build a list of items per invoice.
transactions = (
	df.groupby("InvoiceNo")["Description"]
	.apply(lambda x: list(set(x.astype(str))))
	.tolist()
)

te = TransactionEncoder()
te_array: Any = te.fit(transactions).transform(transactions, sparse=False)
if hasattr(te_array, "toarray"):
	te_array = te_array.toarray()
basket_df = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets = apriori(basket_df, min_support=0.02, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.3)

print("Frequent itemsets (top 10):")
print(frequent_itemsets.sort_values("support", ascending=False).head(10))

print("\nAssociation rules (top 10 by confidence):")
print(rules.sort_values("confidence", ascending=False).head(10))

Frequent itemsets (top 10):
      support                                         itemsets
273  0.112237  frozenset({WHITE HANGING HEART T-LIGHT HOLDER})
102  0.103894             frozenset({JUMBO BAG RED RETROSPOT})
197  0.098778            frozenset({REGENCY CAKESTAND 3 TIER})
157  0.083731                       frozenset({PARTY BUNTING})
127  0.077672             frozenset({LUNCH BAG RED RETROSPOT})
15   0.072259       frozenset({ASSORTED COLOUR BIRD ORNAMENT})
218  0.068782   frozenset({SET OF 3 CAKE TINS PANTRY DESIGN })
149  0.065554     frozenset({PACK OF 72 RETROSPOT CAKE CASES})
120  0.063220             frozenset({LUNCH BAG  BLACK SKULL.})
139  0.062028     frozenset({NATURAL SLATE HEART CHALKBOARD })

Association rules (top 10 by confidence):
                                           antecedents  \
136  frozenset({ROSES REGENCY TEACUP AND SAUCER , P...   
137  frozenset({GREEN REGENCY TEACUP AND SAUCER, PI...   
24         frozenset({PINK REGENCY TEACUP AND SAUCER})   
147 

In [ ]:
from IPython.display import display
import ipywidgets as widgets

# 1) Compact summary of the loaded dataset and Apriori output.
summary_df = pd.DataFrame(
    [
        ("dataset_path", str(dataset_path)),
        ("rows", len(df)),
        ("columns", len(df.columns)),
        ("transactions", len(transactions)),
        ("unique_items", basket_df.shape[1]),
        ("frequent_itemsets", len(frequent_itemsets)),
        ("association_rules", len(rules)),
        ("avg_support", round(float(frequent_itemsets["support"].mean()), 4)),
        ("avg_confidence", round(float(rules["confidence"].mean()), 4) if len(rules) else 0.0),
        ("avg_lift", round(float(rules["lift"].mean()), 4) if len(rules) else 0.0),
    ],
    columns=["metric", "value"],
)
display(summary_df)

# 2) Simple interactive table (Excel-like controls).
rules_view = rules.copy()
for col in ["antecedents", "consequents"]:
    rules_view[col] = rules_view[col].apply(lambda x: ", ".join(sorted(list(x))))

default_cols = ["antecedents", "consequents", "support", "confidence", "lift"]
all_cols = list(rules_view.columns)
numeric_cols = [c for c in all_cols if pd.api.types.is_numeric_dtype(rules_view[c])]

col_picker = widgets.SelectMultiple(
    options=all_cols,
    value=tuple(default_cols),
    description="Columns",
    rows=min(10, len(all_cols)),
)
search_text = widgets.Text(
    value="",
    placeholder="type keyword (e.g. HEART)",
    description="Search",
)
min_conf = widgets.FloatSlider(
    value=0.30, min=0.0, max=1.0, step=0.01, description="Min conf"
 )
min_lift = widgets.FloatSlider(
    value=1.0, min=0.0, max=float(max(1.0, rules_view["lift"].max())) if len(rules_view) else 5.0,
    step=0.1,
    description="Min lift",
)
sort_by = widgets.Dropdown(
    options=numeric_cols if numeric_cols else all_cols,
    value="confidence" if "confidence" in all_cols else all_cols[0],
    description="Sort by",
)
descending = widgets.Checkbox(value=True, description="Descending")
max_rows = widgets.IntSlider(value=20, min=5, max=100, step=5, description="Rows")
out = widgets.Output()

def refresh(_=None):
    with out:
        out.clear_output(wait=True)
        view = rules_view.copy()
        view = view[view["confidence"] >= min_conf.value]
        view = view[view["lift"] >= min_lift.value]

        term = search_text.value.strip().lower()
        if term:
            mask = (
                view["antecedents"].str.lower().str.contains(term, regex=False)
                | view["consequents"].str.lower().str.contains(term, regex=False)
            )
            view = view[mask]

        selected = list(col_picker.value) if col_picker.value else default_cols
        selected = [c for c in selected if c in view.columns]
        if not selected:
            selected = ["antecedents", "consequents", "confidence"]

        if len(view) and sort_by.value in view.columns:
            view = view.sort_values(sort_by.value, ascending=not descending.value)

        print(f"Rows shown: {min(max_rows.value, len(view))} / {len(view)}")
        display(view[selected].head(max_rows.value))

for w in [col_picker, search_text, min_conf, min_lift, sort_by, descending, max_rows]:
    w.observe(refresh, names="value")

display(widgets.VBox([
    widgets.HTML("<b>Interactive Association Rules Table</b>"),
    col_picker,
    search_text,
    widgets.HBox([min_conf, min_lift, max_rows]),
    widgets.HBox([sort_by, descending]),
    out,
]))
refresh()

,metric,value
0,dataset_path,/home/komputer7/SparkProjects/online-retail-py...
1,rows,530693
2,columns,8
3,transactions,20136
4,unique_items,4077
5,frequent_itemsets,375
6,association_rules,151
7,avg_support,0.031
8,avg_confidence,0.4993
9,avg_lift,9.0954
